In [3]:
# -*- coding: utf-8 -*-
"""GRU_Autoencoder_Feature_Extraction.py

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1qn8wXi8x9bLTmcGnQdCL2SvbIQdqp7wS
"""

import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.utils.data as Data
import torch.nn.functional as F

from google.colab import drive
from tqdm.auto import trange


drive.mount('/content/drive')


BASE_DIR = '/content/drive/MyDrive/Insider_Threat_Project'
SAMPLE_DATA_DIR = os.path.join(BASE_DIR, 'sample_data')
RESULT_DIR = os.path.join(BASE_DIR, 'gru_feature_extraction_result')

BEST_MODEL_DIR = os.path.join(RESULT_DIR, 'early_stop_model')
CHECKPOINT_DIR = os.path.join(RESULT_DIR, 'checkpoints')
PLOT_DIR = os.path.join(RESULT_DIR, 'plots')
LOG_DIR = os.path.join(RESULT_DIR, 'logs')

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


SEQUENCE_FILE = os.path.join(
    SAMPLE_DATA_DIR,
    '1-data-test-combine-sequence.csv'
)

MANUAL_FILE = os.path.join(
    SAMPLE_DATA_DIR,
    '1-data-test-combine.csv'
)


if not os.path.exists(SEQUENCE_FILE):
    raise FileNotFoundError(
        f'Cannot find sequence CSV: {SEQUENCE_FILE}'
    )

if not os.path.exists(MANUAL_FILE):
    raise FileNotFoundError(
        f'Cannot find manual-feature CSV: {MANUAL_FILE}'
    )


SEQUENCE_LENGTH = 74
INPUT_SIZE = 25
HIDDEN_SIZE = 100
ENCODE_SIZE = 5
N_FEATURES = 5

EPOCHS = 1000
LEARNING_RATE = 0.01
TRAIN_ROUNDS = 5
BATCH_SIZE = 2000

PATIENCE = 20
DELTA = 0.0001

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', DEVICE)

if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


sequence_data = pd.read_csv(SEQUENCE_FILE)
df_manual = pd.read_csv(MANUAL_FILE)

print('Sequence data shape:', sequence_data.shape)
print('Manual feature data shape:', df_manual.shape)


if len(sequence_data) != len(df_manual):
    raise ValueError(
        'The two CSV files have different row counts. '
        'Do not concatenate their features until their rows match.'
    )


identity_columns = [
    'date_index',
    'user_index',
    'date',
    'label'
]

for col in identity_columns:
    if col not in sequence_data.columns:
        raise ValueError(
            f'{col} is missing from the sequence CSV.'
        )

    if col not in df_manual.columns:
        raise ValueError(
            f'{col} is missing from the manual feature CSV.'
        )


sequence_identity = (
    sequence_data[identity_columns]
    .fillna('')
    .astype(str)
    .reset_index(drop=True)
)

manual_identity = (
    df_manual[identity_columns]
    .fillna('')
    .astype(str)
    .reset_index(drop=True)
)

if not sequence_identity.equals(manual_identity):
    raise ValueError(
        'The rows of the sequence and manual-feature CSVs '
        'do not match in the same order.'
    )

print('Row identity and ordering verified.')


if len(sequence_data.columns) < 4 + SEQUENCE_LENGTH:
    raise ValueError(
        'The sequence CSV does not contain 74 activity columns '
        'after its first four metadata columns.'
    )


activity_columns = list(
    sequence_data.columns[4:4 + SEQUENCE_LENGTH]
)

sequence_data[activity_columns] = (
    sequence_data[activity_columns].fillna(0)
)

activity_values = sequence_data[activity_columns].to_numpy(
    dtype=np.float32
)

if not np.isfinite(activity_values).all():
    raise ValueError(
        'The activity sequences contain infinite or invalid values.'
    )

if not np.allclose(
    activity_values,
    np.round(activity_values)
):
    raise ValueError(
        'Activity values must be integers for one-hot encoding.'
    )

activity_values = np.round(activity_values).astype(np.int64)

if activity_values.min() < 0 or activity_values.max() >= INPUT_SIZE:
    raise ValueError(
        f'Activity values must be between 0 and {INPUT_SIZE - 1}.'
    )


x = torch.tensor(
    activity_values,
    dtype=torch.long
)

x = F.one_hot(
    x,
    num_classes=INPUT_SIZE
).float()

labels = sequence_data['label'].to_numpy(dtype=np.int64)

y = torch.tensor(labels, dtype=torch.int32)

print('Encoded sequence shape:', tuple(x.shape))
print('Labels shape:', tuple(y.shape))
print('Label counts:')
print(pd.Series(labels).value_counts().sort_index())


batch_data = Data.DataLoader(
    dataset=Data.TensorDataset(x, y),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


class GruAutoEncoder(nn.Module):

    def __init__(
        self,
        input_size=25,
        hidden_size=100,
        batch_size=2000,
        encode_size=5
    ):
        super(GruAutoEncoder, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.batch_size = batch_size
        self.encode_size = encode_size

        self.encoder_gru = nn.GRU(
            self.input_size,
            self.hidden_size,
            batch_first=True
        )

        self.encoder_fc = nn.Linear(
            self.hidden_size,
            self.encode_size
        )

        self.decoder_gru = nn.GRU(
            self.hidden_size,
            self.input_size,
            batch_first=True
        )

        self.decoder_fc = nn.Linear(
            self.encode_size,
            self.hidden_size
        )

        self.relu = nn.ReLU()

        self.output_fc = nn.Linear(
            SEQUENCE_LENGTH * self.encode_size,
            N_FEATURES
        )

        self.decoder_input_fc = nn.Linear(
            N_FEATURES,
            SEQUENCE_LENGTH * self.encode_size
        )

    def forward(self, x):

        encoder_gru, _ = self.encoder_gru(x)

        encoder_fc = self.encoder_fc(encoder_gru)

        encoder_out = encoder_fc.flatten(1)

        encoder_out = self.output_fc(encoder_out)

        decoder_input = self.decoder_input_fc(encoder_out)

        decoder_input = decoder_input.view(
            -1,
            SEQUENCE_LENGTH,
            self.encode_size
        )

        decoder_fc = self.decoder_fc(decoder_input)

        decoder_gru, _ = self.decoder_gru(decoder_fc)

        return encoder_out, decoder_gru


class EarlyStopping:

    def __init__(
        self,
        save_path,
        patience=20,
        verbose=True,
        delta=0.0001,
        metric='loss'
    ):
        self.save_path = save_path
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.delta = delta
        self.metric = metric
        self.val_bound = float('inf')
        self.sign = -1 if metric == 'loss' else 1

    def __call__(self, value, model):

        value = float(value)
        score = value * self.sign

        if self.best_score is None:

            self.best_score = score
            self.save_checkpoint(value, model)

        elif score < self.best_score + self.delta:

            self.counter += 1

            print(
                f'EarlyStopping counter: '
                f'{self.counter} out of {self.patience}'
            )

            if self.counter >= self.patience:
                self.early_stop = True

        else:

            self.best_score = score
            self.save_checkpoint(value, model)
            self.counter = 0

    def save_checkpoint(self, value, model):

        if self.verbose:

            if self.sign == -1:
                print(
                    f'Validation {self.metric} decreased '
                    f'({self.val_bound:.6f} --> {value:.6f}). '
                    f'Saving model ...'
                )
            else:
                print(
                    f'Validation {self.metric} increased '
                    f'({self.val_bound:.6f} --> {value:.6f}). '
                    f'Saving model ...'
                )

        torch.save(
            model.state_dict(),
            self.save_path
        )

        self.val_bound = value


def save_training_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    early_stopping,
    train_losses,
    validation_losses,
    round_number,
    finished=False
):

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'round_number': round_number,
        'early_stopping': {
            'counter': early_stopping.counter,
            'best_score': early_stopping.best_score,
            'early_stop': early_stopping.early_stop,
            'val_loss_min': early_stopping.val_loss_min,
            'val_bound': early_stopping.val_bound
        },
        'train_losses': train_losses,
        'validation_losses': validation_losses,
        'finished': finished,
        'torch_rng_state': torch.get_rng_state(),
        'numpy_rng_state': np.random.get_state(),
        'python_rng_state': random.getstate()
    }

    if torch.cuda.is_available():
        checkpoint['cuda_rng_state_all'] = (
            torch.cuda.get_rng_state_all()
        )

    temporary_path = path + '.tmp'

    torch.save(checkpoint, temporary_path)

    os.replace(temporary_path, path)


def load_training_checkpoint(
    path,
    model,
    optimizer,
    early_stopping
):

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint['model_state_dict']
    )

    optimizer.load_state_dict(
        checkpoint['optimizer_state_dict']
    )

    state = checkpoint['early_stopping']

    early_stopping.counter = state['counter']
    early_stopping.best_score = state['best_score']
    early_stopping.early_stop = state['early_stop']
    early_stopping.val_loss_min = state['val_loss_min']
    early_stopping.val_bound = state['val_bound']

    torch.set_rng_state(
        checkpoint['torch_rng_state'].cpu()
    )

    np.random.set_state(
        checkpoint['numpy_rng_state']
    )

    random.setstate(
        checkpoint['python_rng_state']
    )

    if (
        torch.cuda.is_available()
        and 'cuda_rng_state_all' in checkpoint
    ):
        torch.cuda.set_rng_state_all(
            checkpoint['cuda_rng_state_all']
        )

    return checkpoint


def save_loss_plot(
    train_losses,
    validation_losses,
    output_path
):

    plt.figure(figsize=(9, 5))

    epochs_completed = range(1, len(train_losses) + 1)

    plt.plot(
        epochs_completed,
        train_losses,
        label='Training loss'
    )

    plt.plot(
        epochs_completed,
        validation_losses,
        label='Monitoring loss'
    )

    if validation_losses:
        best_epoch = int(
            np.argmin(validation_losses)
        ) + 1

        plt.axvline(
            best_epoch,
            linestyle='--',
            label='Lowest monitoring loss'
        )

    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('GRU Autoencoder Training')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=200
    )

    plt.close()


start_time = time.time()

round_results = []

for r in range(TRAIN_ROUNDS):

    print('\n' + '=' * 60)
    print(f'TRAINING ROUND {r}')
    print('=' * 60)

    best_model_path = os.path.join(
        BEST_MODEL_DIR,
        f'gru_seq{SEQUENCE_LENGTH}_{r}round_best.pt'
    )

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f'gru_seq{SEQUENCE_LENGTH}_{r}round_latest.pt'
    )

    complete_path = os.path.join(
        LOG_DIR,
        f'gru_seq{SEQUENCE_LENGTH}_{r}round_complete.json'
    )

    history_path = os.path.join(
        LOG_DIR,
        f'gru_seq{SEQUENCE_LENGTH}_{r}round_history.csv'
    )

    plot_path = os.path.join(
        PLOT_DIR,
        f'gru_seq{SEQUENCE_LENGTH}_{r}round_loss.png'
    )

    if os.path.exists(complete_path):

        with open(complete_path, 'r') as f:
            completed_result = json.load(f)

        if os.path.exists(best_model_path):

            print(
                f'Round {r} was already completed. '
                'Loading its saved results.'
            )

            round_results.append(completed_result)
            continue

    model = GruAutoEncoder(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        batch_size=BATCH_SIZE,
        encode_size=ENCODE_SIZE
    ).to(DEVICE)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    early_stopping = EarlyStopping(
        save_path=best_model_path,
        verbose=True,
        patience=PATIENCE,
        delta=DELTA,
        metric='loss'
    )

    train_losses = []
    validation_losses = []
    start_epoch = 0

    if os.path.exists(checkpoint_path):

        print('Found a previous checkpoint.')
        print('Resuming the interrupted round.')

        checkpoint = load_training_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            early_stopping
        )

        train_losses = checkpoint['train_losses']
        validation_losses = checkpoint['validation_losses']

        start_epoch = checkpoint['epoch']

        if checkpoint.get('finished', False):

            print('This round was already marked as finished.')

            if os.path.exists(best_model_path):

                result = {
                    'round': r,
                    'min_loss': float(min(validation_losses)),
                    'stop_epoch': len(validation_losses)
                }

                with open(complete_path, 'w') as f:
                    json.dump(result, f)

                round_results.append(result)
                continue

    for epoch in trange(
        start_epoch,
        EPOCHS,
        desc=f'Round {r}'
    ):

        model.train()

        epoch_batch_losses = []

        for b_x, b_y in batch_data:

            b_x = b_x.to(DEVICE)

            optimizer.zero_grad()

            _, y_pred = model(b_x)

            loss = loss_function(
                y_pred,
                b_x
            )

            loss.backward()

            optimizer.step()

            epoch_batch_losses.append(
                loss.item()
            )

        train_loss = float(
            epoch_batch_losses[-1]
        )

        train_losses.append(train_loss)

        model.eval()

        monitoring_data = x[-500:].to(DEVICE)

        with torch.no_grad():

            _, monitoring_prediction = model(
                monitoring_data
            )

            monitoring_loss = loss_function(
                monitoring_prediction,
                monitoring_data
            ).item()

        validation_losses.append(
            float(monitoring_loss)
        )

        early_stopping(
            monitoring_loss,
            model
        )

        current_epoch = epoch + 1

        save_training_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            current_epoch,
            early_stopping,
            train_losses,
            validation_losses,
            r,
            finished=early_stopping.early_stop
        )

        if current_epoch % 10 == 0:

            print(
                f'Round {r} | '
                f'Epoch {current_epoch} | '
                f'Train loss: {train_loss:.6f} | '
                f'Monitoring loss: {monitoring_loss:.6f}'
            )

        if early_stopping.early_stop:

            print(
                f'Early stopping at epoch {current_epoch}'
            )

            break

    save_loss_plot(
        train_losses,
        validation_losses,
        plot_path
    )

    history_df = pd.DataFrame({
        'epoch': range(1, len(train_losses) + 1),
        'training_loss': train_losses,
        'monitoring_loss': validation_losses
    })

    history_df.to_csv(
        history_path,
        index=False
    )

    result = {
        'round': r,
        'min_loss': float(min(validation_losses)),
        'stop_epoch': len(validation_losses)
    }

    with open(complete_path, 'w') as f:
        json.dump(result, f)

    round_results.append(result)

    print(f'Round {r} completed.')
    print(f'Lowest monitoring loss: {result["min_loss"]:.6f}')
    print(f'Epochs completed: {result["stop_epoch"]}')
    print(f'Best model saved at: {best_model_path}')


perform = pd.DataFrame(round_results)

perform = perform.sort_values(
    'round'
).reset_index(drop=True)

perform.to_csv(
    os.path.join(
        RESULT_DIR,
        f'gru_perform_result_seq{SEQUENCE_LENGTH}.csv'
    ),
    index=False
)

print('\nResults of all training rounds:')
print(perform)

best_round = int(
    perform.loc[
        perform['min_loss'].idxmin(),
        'round'
    ]
)

best_model_path = os.path.join(
    BEST_MODEL_DIR,
    f'gru_seq{SEQUENCE_LENGTH}_{best_round}round_best.pt'
)

print('\nBest round:', best_round)
print('Best model path:', best_model_path)


model_extract = GruAutoEncoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    batch_size=BATCH_SIZE,
    encode_size=ENCODE_SIZE
).to(DEVICE)

model_extract.load_state_dict(
    torch.load(
        best_model_path,
        map_location=DEVICE,
        weights_only=True
    )
)

model_extract.eval()

with torch.no_grad():

    output, _ = model_extract(
        x.to(DEVICE)
    )

output = output.cpu().numpy()

df_feature = pd.DataFrame(
    output,
    columns=[f'gru_feature_{i + 1}' for i in range(N_FEATURES)]
)

if len(df_feature) != len(df_manual):
    raise ValueError(
        'Extracted feature rows do not match manual-feature rows.'
    )

df_save = pd.concat(
    [
        df_manual.reset_index(drop=True),
        df_feature.reset_index(drop=True)
    ],
    axis=1
)

feature_path = os.path.join(
    RESULT_DIR,
    f'1-data-test-manual+gru_seq{SEQUENCE_LENGTH}.csv'
)

df_save.to_csv(
    feature_path,
    index=False
)

print('\nFeature extraction completed.')
print('Extracted GRU feature shape:', df_feature.shape)
print('Combined feature shape:', df_save.shape)
print('Combined features saved to:', feature_path)

print('\nTotal time:', round(time.time() - start_time, 2), 'seconds')
print('\nAll results are stored in Google Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
Sequence data shape: (1908, 103)
Manual feature data shape: (1908, 10)
Row identity and ordering verified.
Encoded sequence shape: (1908, 74, 25)
Labels shape: (1908,)
Label counts:
0    954
1    954
Name: count, dtype: int64

TRAINING ROUND 0


Round 0:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.054727). Saving model ...
Validation loss decreased (0.054727 --> 0.050918). Saving model ...
Validation loss decreased (0.050918 --> 0.035643). Saving model ...
Validation loss decreased (0.035643 --> 0.031172). Saving model ...
Validation loss decreased (0.031172 --> 0.028779). Saving model ...
Validation loss decreased (0.028779 --> 0.026439). Saving model ...
Validation loss decreased (0.026439 --> 0.022923). Saving model ...
Validation loss decreased (0.022923 --> 0.019621). Saving model ...
Validation loss decreased (0.019621 --> 0.017181). Saving model ...
Validation loss decreased (0.017181 --> 0.015432). Saving model ...
Round 0 | Epoch 10 | Train loss: 0.018169 | Monitoring loss: 0.015432
Validation loss decreased (0.015432 --> 0.014552). Saving model ...
EarlyStopping counter: 1 out of 20
EarlyStopping counter: 2 out of 20
EarlyStopping counter: 3 out of 20
Validation loss decreased (0.014552 --> 0.014325). Saving model ...
Validation los

Round 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.056274). Saving model ...
Validation loss decreased (0.056274 --> 0.038488). Saving model ...
Validation loss decreased (0.038488 --> 0.037515). Saving model ...
Validation loss decreased (0.037515 --> 0.037037). Saving model ...
Validation loss decreased (0.037037 --> 0.032373). Saving model ...
Validation loss decreased (0.032373 --> 0.025790). Saving model ...
Validation loss decreased (0.025790 --> 0.020356). Saving model ...
Validation loss decreased (0.020356 --> 0.017552). Saving model ...
Validation loss decreased (0.017552 --> 0.016714). Saving model ...
EarlyStopping counter: 1 out of 20
Round 1 | Epoch 10 | Train loss: 0.017762 | Monitoring loss: 0.016839
EarlyStopping counter: 2 out of 20
EarlyStopping counter: 3 out of 20
Validation loss decreased (0.016714 --> 0.016013). Saving model ...
Validation loss decreased (0.016013 --> 0.014821). Saving model ...
Validation loss decreased (0.014821 --> 0.013898). Saving model ...
Validation los

Round 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.072735). Saving model ...
Validation loss decreased (0.072735 --> 0.040685). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.040685 --> 0.037041). Saving model ...
Validation loss decreased (0.037041 --> 0.035885). Saving model ...
Validation loss decreased (0.035885 --> 0.029587). Saving model ...
Validation loss decreased (0.029587 --> 0.022281). Saving model ...
Validation loss decreased (0.022281 --> 0.017735). Saving model ...
Validation loss decreased (0.017735 --> 0.015889). Saving model ...
EarlyStopping counter: 1 out of 20
Round 2 | Epoch 10 | Train loss: 0.016974 | Monitoring loss: 0.015802
EarlyStopping counter: 2 out of 20
EarlyStopping counter: 3 out of 20
EarlyStopping counter: 4 out of 20
EarlyStopping counter: 5 out of 20
Validation loss decreased (0.015889 --> 0.015426). Saving model ...
Validation loss decreased (0.015426 --> 0.014252). Saving model ...
Validation loss decreased (0.014252 --> 0.013

Round 3:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.070168). Saving model ...
Validation loss decreased (0.070168 --> 0.048280). Saving model ...
Validation loss decreased (0.048280 --> 0.035527). Saving model ...
EarlyStopping counter: 1 out of 20
EarlyStopping counter: 2 out of 20
Validation loss decreased (0.035527 --> 0.031809). Saving model ...
Validation loss decreased (0.031809 --> 0.026240). Saving model ...
Validation loss decreased (0.026240 --> 0.020409). Saving model ...
Validation loss decreased (0.020409 --> 0.016129). Saving model ...
Validation loss decreased (0.016129 --> 0.014485). Saving model ...
Round 3 | Epoch 10 | Train loss: 0.017219 | Monitoring loss: 0.014485
EarlyStopping counter: 1 out of 20
EarlyStopping counter: 2 out of 20
EarlyStopping counter: 3 out of 20
EarlyStopping counter: 4 out of 20
EarlyStopping counter: 5 out of 20
EarlyStopping counter: 6 out of 20
Validation loss decreased (0.014485 --> 0.013640). Saving model ...
Validation loss decreased (0.013640 --> 0.0

Round 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.075596). Saving model ...
Validation loss decreased (0.075596 --> 0.048800). Saving model ...
Validation loss decreased (0.048800 --> 0.039723). Saving model ...
Validation loss decreased (0.039723 --> 0.029532). Saving model ...
Validation loss decreased (0.029532 --> 0.024147). Saving model ...
Validation loss decreased (0.024147 --> 0.021639). Saving model ...
Validation loss decreased (0.021639 --> 0.020530). Saving model ...
Validation loss decreased (0.020530 --> 0.019985). Saving model ...
Validation loss decreased (0.019985 --> 0.019816). Saving model ...
Validation loss decreased (0.019816 --> 0.019623). Saving model ...
Round 4 | Epoch 10 | Train loss: 0.020833 | Monitoring loss: 0.019623
Validation loss decreased (0.019623 --> 0.018407). Saving model ...
Validation loss decreased (0.018407 --> 0.016193). Saving model ...
Validation loss decreased (0.016193 --> 0.014024). Saving model ...
Validation loss decreased (0.014024 --> 0.012846). 